In [ ]:
ANTHROPIC_API_KEY= ""

In [ ]:
import os
from anthropic import Anthropic

os.environ["ANTHROPIC_API_KEY"] = ""
clientA = Anthropic()

In [3]:

import os, getpass
import os
from anthropic import Anthropic
import sqlite3
import re
import uuid
import json

LIVE = bool(ANTHROPIC_API_KEY)
MODEL = 'claude-sonnet-4-6'
print('LIVE mode' if LIVE else 'OFFLINE mock mode')

LIVE mode


In [4]:
db = sqlite3.connect(':memory:', check_same_thread=False)

db.execute('CREATE TABLE inventory (sku TEXT PRIMARY KEY, name TEXT, qty INTEGER, price REAL)')
db.execute('CREATE TABLE orders (id INTEGER PRIMARY KEY AUTOINCREMENT, sku TEXT, qty INTEGER, total REAL, status TEXT)')

db.executemany('INSERT INTO inventory VALUES (?,?,?,?)', [
    ('KB-01', 'Mechanical keyboard', 12, 129.0),
    ('HUB-2', 'USB-C hub',           0,  58.0),
    ('MON-4', '4K monitor',          5, 410.0),
])

db.commit()
print('inventory seeded')


# ============================================================
# Step 3 — Redis Stream queue + DLQ config
# ============================================================

import fakeredis

r = fakeredis.FakeStrictRedis()

STREAM = 'emails'
GROUP = 'mailers'
DLQ_STREAM = 'emails:dlq'

MAX_EMAIL_RETRIES = int(os.environ.get("MAX_EMAIL_RETRIES", "3"))
EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

def is_valid_email(addr: str) -> bool:
    return bool(addr and EMAIL_RE.match(addr))

try:
    r.xgroup_create(STREAM, GROUP, id='0', mkstream=True)
except Exception as e:
    print('group exists:', e)

def enqueue_email(to, subject, body, retries=0, job_id=None):
    """
    Add an email job to the stream.
    On retries, keep the same job_id so check_job stays stable.
    """
    if job_id is None:
        job_id = uuid.uuid4().hex[:8]

    r.xadd(STREAM, {
        'job_id': job_id,
        'to': to,
        'subject': subject,
        'body': body,
        'retries': str(retries)
    })

    if retries == 0:
        r.set(f'jobresult:{job_id}', 'queued')

    return job_id

print('queue ready; sample job id ->', enqueue_email('a@x.com', 'hi', 'test'))


# ============================================================
# Step 4 — Worker with retry + DLQ
# ============================================================

def run_worker(max_msgs=10, worker_name='worker-1'):
    """
    Reads jobs from Redis stream.

    Success:
      - validate email
      - mark sent
      - ack

    Failure:
      - retry until MAX_EMAIL_RETRIES
      - after that move to DLQ and ack original msg
    """
    processed = 0
    resp = r.xreadgroup(GROUP, worker_name, {STREAM: '>'}, count=max_msgs)

    for _stream, msgs in resp or []:
        for msg_id, fields in msgs:
            f = {k.decode(): v.decode() for k, v in fields.items()}

            job_id  = f.get('job_id')
            to      = f.get('to', '')
            subject = f.get('subject', '')
            body    = f.get('body', '')
            retries = int(f.get('retries', '0'))

            try:
                if not is_valid_email(to):
                    raise ValueError(f'invalid email address: {to}')

                # success path
                r.set(f'jobresult:{job_id}', 'sent')
                r.xack(STREAM, GROUP, msg_id)
                processed += 1

            except Exception as e:
                next_retry = retries + 1

                if next_retry >= MAX_EMAIL_RETRIES:
                    # move to DLQ
                    r.xadd(DLQ_STREAM, {
                        'job_id': job_id,
                        'to': to,
                        'subject': subject,
                        'body': body,
                        'retries': str(next_retry),
                        'error': str(e),
                        'failed_stream': STREAM,
                        'failed_msg_id': msg_id.decode() if isinstance(msg_id, bytes) else str(msg_id)
                    })
                    r.set(f'jobresult:{job_id}', f'dlq:{e}')
                    r.xack(STREAM, GROUP, msg_id)
                else:
                    # requeue with same job_id, incremented retry count
                    enqueue_email(
                        to=to,
                        subject=subject,
                        body=body,
                        retries=next_retry,
                        job_id=job_id
                    )
                    r.set(
                        f'jobresult:{job_id}',
                        f'retrying ({next_retry}/{MAX_EMAIL_RETRIES - 1}): {e}'
                    )
                    r.xack(STREAM, GROUP, msg_id)

                processed += 1

    return processed

print('worker processed', run_worker(), 'job(s)')


# ============================================================
# Step 5 — Tool functions
# ============================================================

def check_inventory(sku: str):
    row = db.execute(
        'SELECT sku,name,qty,price FROM inventory WHERE sku=? LIMIT 1',
        (sku,)
    ).fetchone()

    if not row:
        return {'error': f'unknown sku {sku}'}

    return {'sku': row[0], 'name': row[1], 'qty': row[2], 'price': row[3]}


def create_order(sku: str, qty: int):
    cur = db.execute(
        'SELECT qty,price FROM inventory WHERE sku=? LIMIT 1',
        (sku,)
    ).fetchone()

    if not cur:
        return {'error': f'unknown sku {sku}'}

    have, price = cur

    if qty <= 0:
        return {'error': 'qty must be positive'}

    if have < qty:
        return {'error': f'insufficient stock: have {have}, need {qty}'}

    db.execute('UPDATE inventory SET qty=qty-? WHERE sku=?', (qty, sku))
    cur2 = db.execute(
        'INSERT INTO orders (sku,qty,total,status) VALUES (?,?,?,?)',
        (sku, qty, round(price * qty, 2), 'created')
    )
    db.commit()

    return {
        'order_id': cur2.lastrowid,
        'sku': sku,
        'qty': qty,
        'total': round(price * qty, 2)
    }


def verify_order(order_id: int):
    """
    Evaluator / verifier tool:
    Re-read the order from DB and confirm total + status.
    """
    row = db.execute(
        'SELECT id, sku, qty, total, status FROM orders WHERE id=? LIMIT 1',
        (order_id,)
    ).fetchone()

    if not row:
        return {'error': f'order {order_id} not found'}

    return {
        'order_id': row[0],
        'sku': row[1],
        'qty': row[2],
        'total': row[3],
        'status': row[4],
        'verified': True
    }


def send_confirmation(to: str, order_id: int):
    job_id = enqueue_email(
        to,
        f'Order {order_id} confirmed',
        f'Your order {order_id} is on its way.'
    )
    return {'job_id': job_id, 'status': 'queued'}


def check_job(job_id: str):
    v = r.get(f'jobresult:{job_id}')
    return {'job_id': job_id, 'status': v.decode() if v else 'unknown'}


def inspect_dlq(limit: int = 10):
    """
    Inspect latest failed email jobs from DLQ.
    """
    if limit <= 0:
        return {'error': 'limit must be positive'}

    rows = r.xrevrange(DLQ_STREAM, count=limit)

    items = []
    for msg_id, fields in rows:
        f = {k.decode(): v.decode() for k, v in fields.items()}
        items.append({
            'stream_id': msg_id.decode() if isinstance(msg_id, bytes) else str(msg_id),
            'job_id': f.get('job_id'),
            'to': f.get('to'),
            'subject': f.get('subject'),
            'retries': int(f.get('retries', '0')),
            'error': f.get('error'),
            'failed_stream': f.get('failed_stream'),
            'failed_msg_id': f.get('failed_msg_id')
        })

    return {'count': len(items), 'items': items}


# quick manual chain
inv = check_inventory('KB-01')
print(inv)

od = create_order('KB-01', 2)
print(od)

ver = verify_order(od['order_id'])
print('verify:', ver)

jb = send_confirmation('asha@x.com', od['order_id'])
print(jb)

run_worker()
print(check_job(jb['job_id']))


# ============================================================
# Step 6 — Tool schemas + dispatch
# ============================================================

TOOLS = [
    {
        'name': 'check_inventory',
        'description': 'Check stock and price for a product SKU before ordering.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'sku': {
                    'type': 'string',
                    'description': 'Product SKU, e.g. KB-01.'
                }
            },
            'required': ['sku']
        }
    },
    {
        'name': 'create_order',
        'description': 'Create an order for a SKU and quantity. Fails if stock is insufficient.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'sku': {'type': 'string'},
                'qty': {'type': 'integer', 'description': 'Units to order (>0).'}
            },
            'required': ['sku', 'qty']
        }
    },
    {
        'name': 'verify_order',
        'description': 'Re-read an order from the database and verify its total and status before declaring success.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'order_id': {'type': 'integer'}
            },
            'required': ['order_id']
        }
    },
    {
        'name': 'send_confirmation',
        'description': 'Queue a confirmation email for a created order. Returns a job_id immediately.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'to': {'type': 'string', 'description': 'Customer email.'},
                'order_id': {'type': 'integer'}
            },
            'required': ['to', 'order_id']
        }
    },
    {
        'name': 'check_job',
        'description': 'Check the status of a queued email job by job_id.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'job_id': {'type': 'string'}
            },
            'required': ['job_id']
        }
    },
    {
        'name': 'inspect_dlq',
        'description': 'Inspect failed email jobs that were moved to the dead-letter queue.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'limit': {
                    'type': 'integer',
                    'description': 'Maximum number of failed jobs to return.'
                }
            },
            'required': []
        }
    }
]

DISPATCH = {
    'check_inventory': check_inventory,
    'create_order': create_order,
    'verify_order': verify_order,
    'send_confirmation': send_confirmation,
    'check_job': check_job,
    'inspect_dlq': inspect_dlq
}

def run_tool(name, args):
    fn = DISPATCH.get(name)
    if not fn:
        return {'error': f'unknown tool {name}'}, True

    try:
        out = fn(**args)
        return out, isinstance(out, dict) and 'error' in out
    except Exception as e:
        return {'error': repr(e)}, True

print('tools ready')


# ============================================================
# Step 7 — Orchestrating agent loop (with evaluator requirement)
# ============================================================

SYSTEM = (
    'You are an ordering agent. '
    'To place an order successfully you MUST follow this sequence: '
    '1) check_inventory, '
    '2) create_order, '
    '3) send_confirmation, '
    '4) verify_order using the returned order_id, '
    'and only after verify_order confirms the order may you declare success. '
    'Report the verified order total, order status, and the email job status. '
    'If stock is insufficient, say so and do not create the order. '
    'If a user asks about failed email jobs, use inspect_dlq.'
)

def agent(user_text, max_steps=10, verbose=True):
    if not LIVE:
        # offline mock branch for testing without Claude key
        if verbose:
            print('… (mock) chaining check_inventory → create_order → send_confirmation → verify_order')

        inv, _ = run_tool('check_inventory', {'sku': 'MON-4'})
        od, _ = run_tool('create_order', {'sku': 'MON-4', 'qty': 1})
        jb, _ = run_tool('send_confirmation', {'to': 'asha@x.com', 'order_id': od['order_id']})
        run_worker()
        st, _ = run_tool('check_job', {'job_id': jb['job_id']})
        vr, _ = run_tool('verify_order', {'order_id': od['order_id']})

        return (
            f"(mock) Verified order {vr['order_id']} total ${vr['total']} "
            f"status={vr['status']}; email {st['status']}."
        )

    from anthropic import Anthropic
    A = Anthropic()
    messages = [{'role': 'user', 'content': user_text}]

    for _ in range(max_steps):
        resp = A.messages.create(
            model=MODEL,
            max_tokens=1024,
            system=SYSTEM,
            tools=TOOLS,
            messages=messages
        )

        if resp.stop_reason == 'tool_use':
            messages.append({
                'role': 'assistant',
                'content': [b.model_dump() for b in resp.content]
            })

            results = []
            for b in resp.content:
                if b.type == 'tool_use':
                    if verbose:
                        print(f'  → {b.name}({b.input})')

                    out, is_err = run_tool(b.name, b.input)
                    results.append({
                        'type': 'tool_result',
                        'tool_use_id': b.id,
                        'content': json.dumps(out),
                        'is_error': is_err
                    })

            messages.append({'role': 'user', 'content': results})

            # drain email queue between steps
            run_worker()
            continue

        return ''.join(b.text for b in resp.content if b.type == 'text')

    return '(max steps reached)'


print(agent('Order one 4K monitor (SKU MON-4) and email asha@x.com the confirmation.'))


# ============================================================
# Step 8 — Show resulting state
# ============================================================

print('\nOrders table:')
for row in db.execute('SELECT id,sku,qty,total,status FROM orders').fetchall():
    print(' ', row)

print('\nRemaining stock:')
for row in db.execute('SELECT sku,qty FROM inventory').fetchall():
    print(' ', row)

print('\nOut-of-stock attempt:')
print(run_tool('create_order', {'sku': 'HUB-2', 'qty': 1}))







inventory seeded
queue ready; sample job id -> f4ea74a1
worker processed 1 job(s)
{'sku': 'KB-01', 'name': 'Mechanical keyboard', 'qty': 12, 'price': 129.0}
{'order_id': 1, 'sku': 'KB-01', 'qty': 2, 'total': 258.0}
verify: {'order_id': 1, 'sku': 'KB-01', 'qty': 2, 'total': 258.0, 'status': 'created', 'verified': True}
{'job_id': '1fe72527', 'status': 'queued'}
{'job_id': '1fe72527', 'status': 'sent'}
tools ready
  → check_inventory({'sku': 'MON-4'})
  → create_order({'sku': 'MON-4', 'qty': 1})
  → send_confirmation({'to': 'asha@x.com', 'order_id': 2})
  → verify_order({'order_id': 2})
  → check_job({'job_id': '7a8d54d9'})
Everything went through successfully! Here's the full summary:

| Detail | Value |
|---|---|
| **Product** | 4K Monitor (MON-4) |
| **Quantity** | 1 |
| **Order ID** | 2 |
| **Order Total** | $410.00 |
| **Order Status** | ✅ Created & Verified |
| **Confirmation Email** | ✅ Sent to asha@x.com |

The order is confirmed and Asha's confirmation email has been successfull

In [5]:
# ============================================================
# Step 9 — TEST CALLS for verify_order only
# ============================================================

print('\n' + '='*72)
print('TEST 1 — Create an order and verify it')
print('='*72)

# create one order first
od = create_order('MON-4', 1)
print('order created:', od)

# now verify that order from DB
verification = verify_order(od['order_id'])
print('verification:', verification)


print('\n' + '='*72)
print('TEST 2 — Verify another valid order')
print('='*72)

od2 = create_order('KB-01', 2)
print('order created:', od2)

verification2 = verify_order(od2['order_id'])
print('verification:', verification2)


print('\n' + '='*72)
print('TEST 3 — verify_order with invalid / missing order id')
print('='*72)

print(verify_order(99999))


TEST 1 — Create an order and verify it
order created: {'order_id': 3, 'sku': 'MON-4', 'qty': 1, 'total': 410.0}
verification: {'order_id': 3, 'sku': 'MON-4', 'qty': 1, 'total': 410.0, 'status': 'created', 'verified': True}

TEST 2 — Verify another valid order
order created: {'order_id': 4, 'sku': 'KB-01', 'qty': 2, 'total': 258.0}
verification: {'order_id': 4, 'sku': 'KB-01', 'qty': 2, 'total': 258.0, 'status': 'created', 'verified': True}

TEST 3 — verify_order with invalid / missing order id
{'error': 'order 99999 not found'}
